# Module 1: University Data Collection

Output: `university_raw_data.csv` (22,117 rows × 38 columns, one row per university per year, 2017–2026)

## Setup

In [ ]:
import re
import pandas as pd

A few small helpers used to clean up messy raw values before comparing/merging them across sources.

In [ ]:
def clean_name(name):
    if not isinstance(name, str):     
        return ""
    name = re.sub(r"\s*\([^)]*\)", "", name)   
    name = re.sub(r"\s+", " ", name).strip()   # collapse extra spaces
    return name


def to_number(value):
    if pd.isna(value):       # Convert strings like '3,730' or '48%' into a plain float
        return None
    text = str(value).replace(",", "").replace("%", "").strip()
    try:
        return float(text)
    except ValueError:
        return None


def split_ratio(ratio_text):
    if pd.isna(ratio_text):   # Convert a 'Female to Male Ratio' string like '41:59' into (41.0, 59.0)
        return None, None
    match = re.match(r"(\d+)\s*:\s*(\d+)", str(ratio_text))
    if not match:
        return None, None
    return float(match.group(1)), float(match.group(2))


def read_csv_safe(path):
    for encoding in ("utf-8-sig", "utf-8", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin1")

## Step 1: Load the QS Ranking datasets

In [ ]:
def load_qs_2017_2022():
    df = read_csv_safe(f"{RAW_FOLDER}/qs-world-university-rankings-2017-to-2022-V2.csv")
    return pd.DataFrame({
        "university_raw": df["university"],
        "year": df["year"],
        "world_rank": df["rank_display"].apply(to_number),
        "overall_score": df["score"].apply(to_number),
        "country": df["country"],
        "city": df["city"],
        "region": df["region"],
        "university_type": df["type"],
        "size": df["size"],
    })


def load_qs_2023():
    df = read_csv_safe(f"{RAW_FOLDER}/2023_QS_World_University_Rankings.csv")
    return pd.DataFrame({
        "university_raw": df["Institution"],
        "year": 2023,
        "world_rank": df["Rank"].apply(to_number),
        "overall_score": df["ScoreScaled"].apply(to_number),
        "country": df["Location"],
        "academic_reputation_score": df["ArScore"].apply(to_number),
        "employer_reputation_score": df["ErScore"].apply(to_number),
        "citations_score": df["CpfScore"].apply(to_number),
    })

In [ ]:
def load_qs_2024():
    df = read_csv_safe(f"{RAW_FOLDER}/2024_QS_World_University_Rankings_1_1__For_qs_com___1_.csv")
    df = df.iloc[1:].reset_index(drop=True)  # first row is a metadata row, not a university
    return pd.DataFrame({
        "university_raw": df["Institution Name"],
        "year": 2024,
        "world_rank": df["2024 RANK"].apply(to_number),
        "overall_score": df["Overall SCORE"].apply(to_number),
        "country": df["Country"],
        "size": df["SIZE"],
        "university_type": df["STATUS"],
        "academic_reputation_score": df["Academic Reputation Score"].apply(to_number),
        "employer_reputation_score": df["Employer Reputation Score"].apply(to_number),
        "citations_score": df["Citations per Faculty Score"].apply(to_number),
    })


def load_qs_2025():
    df = read_csv_safe(f"{RAW_FOLDER}/QS_World_University_Rankings_2025__Top_global_universities_.csv")
    return pd.DataFrame({
        "university_raw": df["Institution_Name"],
        "year": 2025,
        "world_rank": df["RANK_2025"].apply(to_number),
        "overall_score": df["Overall_Score"].apply(to_number),
        "country": df["Location"],
        "region": df["Region"],
        "size": df["SIZE"],
        "academic_reputation_score": df["Academic_Reputation_Score"].apply(to_number),
        "employer_reputation_score": df["Employer_Reputation_Score"].apply(to_number),
        "citations_score": df["Citations_per_Faculty_Score"].apply(to_number),
    })


def load_qs_2026():
    df = read_csv_safe(f"{RAW_FOLDER}/2026_QS_World_University_Rankings.csv")
    return pd.DataFrame({
        "university_raw": df["Institution Name"],
        "year": 2026,
        "world_rank": df["2026 Rank"].apply(to_number),
        "overall_score": df["Overall SCORE"].apply(to_number),
        "country": df["Country/Territory"],
        "region": df["Region"],
        "size": df["Size"],
        "university_type": df["Status"],
        "academic_reputation_score": df["AR SCORE"].apply(to_number),
        "employer_reputation_score": df["ER SCORE"].apply(to_number),
        "citations_score": df["CPF SCORE"].apply(to_number),
    })

## Step 2: Load the World University Ranking (THE) dataset
This adds performance indicators QS doesn't provide, such as student
population, staff ratio, and gender split.

In [ ]:
def load_the():
    df = read_csv_safe(f"{RAW_FOLDER}/THE_World_University_Rankings_2016-2026.csv")
    female_pct, male_pct = zip(*df["Female to Male Ratio"].apply(split_ratio))
    return pd.DataFrame({
        "university_raw": df["Name"],
        "year": df["Year"],
        "world_rank_the": df["Rank"].apply(to_number),
        "overall_score_the": df["Overall Score"].apply(to_number),
        "country": df["Country"],
        "total_students": df["Student Population"].apply(to_number),
        "student_staff_ratio": df["Students to Staff Ratio"].apply(to_number),
        "international_student_ratio": df["International Students"].apply(to_number),
        "female_percentage": female_pct,
        "male_percentage": male_pct,
        "teaching_score": df["Teaching"],
        "research_quality_score": df["Research Quality"],
        "industry_impact_score": df["Industry Impact"],
        "international_outlook_score": df["International Outlook"],
    })

## Step 3: Merge everything into one common structure
All five QS years are stacked together first, then merged with THE on
university name + year.

In [ ]:
qs_all = pd.concat(
    [load_qs_2017_2022(), load_qs_2023(), load_qs_2024(), load_qs_2025(), load_qs_2026()],
    ignore_index=True,
)
qs_all["university_name"] = qs_all["university_raw"].apply(clean_name)

the_all = load_the()
the_all["university_name"] = the_all["university_raw"].apply(clean_name)

master = qs_all.merge(
    the_all.drop(columns=["university_raw"]),
    on=["university_name", "year"],
    how="outer",
    suffixes=("", "_the"),
)

# Use QS's country when we have it, otherwise fall back to THE's country
master["country"] = master["country"].combine_first(master["country_the"])
master = master.drop(columns=["university_raw", "country_the"])

# Put the name column first for readability
master = master[["university_name"] + [c for c in master.columns if c != "university_name"]]
master.head()

## Step 4: Check integration, then save
A quick look at row counts and column completeness before saving the
final Module 1 deliverable.

In [ ]:
print(f"Rows collected: {master.shape[0]}")
print(f"Unique universities: {master['university_name'].nunique()}")

completeness = 100 - master.isna().mean() * 100
print("\nColumn completeness (%):")
print(completeness.round(1).sort_values())

In [ ]:
master.to_csv("university_raw_data.csv", index=False)
print(f"Saved university_raw_data.csv - {master.shape[0]} rows x {master.shape[1]} columns")